# AdaptCLIP Corruption Single-Class Test

RunPod/GPU에서 corruption을 severity 3으로 고정하고, `TARGET_CLASS`만 바꿔 한 클래스씩 가볍게 실험하는 노트북입니다.

- `DATASET`은 `"mvtec"` 또는 `"visa"`로 바꿉니다.
- `TARGET_CLASS`에 한 클래스 이름만 넣으면 그 클래스만 평가합니다.
- 결과는 `result_adaptclip/corruption/<dataset>/<class>/<corruption>_s3` 아래 저장됩니다.


In [9]:
from __future__ import annotations

import os
import re
import shlex
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import torch

REPO = Path.cwd().resolve()
assert (REPO / "test.py").exists(), f"이 노트북은 AdaptCLIP repo root에서 실행해야 합니다: {REPO}"

print("repo:", REPO)
print("torch:", torch.__version__)
print("mps available:", torch.backends.mps.is_available())

repo: /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP
torch: 2.9.0
mps available: True


## 의존성 확인

현재 주피터 커널 환경에 `test.py` 실행에 필요한 패키지가 없으면 설치합니다.

In [10]:
import importlib.util

REQUIRED_PACKAGES = {
    "torchmetrics": "torchmetrics==1.7.3",
    "kornia": "kornia==0.6.3",
    "omegaconf": "omegaconf==2.3.0",
    "cv2": "opencv-python-headless==4.12.0.88",
    "skimage": "scikit-image==0.25.2",
    "sklearn": "scikit-learn==1.7.2",
    "scipy": "scipy==1.13.1",
    "tabulate": "tabulate==0.9.0",
    "timm": "timm==1.0.16",
    "einops": "einops==0.7.0",
    "ftfy": "ftfy==6.3.1",
    "regex": "regex==2023.12.25",
    "tifffile": "tifffile==2025.5.10",
}

missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are available.")

Installing missing packages: ['kornia==0.6.3']



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


## 실험 설정

`CHECKPOINT_PATH`만 실제 pretrained checkpoint 위치에 맞으면 바로 실행됩니다. 비워두면 흔한 경로를 자동 탐색합니다.

In [11]:
# Dataset: "mvtec" 또는 "visa"
DATASET = "mvtec"
TARGET_CLASS = "bottle"  # 여기만 바꿔서 한 클래스씩 실행하세요. 예: "candle", "capsule"
TEST_DATA_PATH = REPO / "dataset" / ("MVTec" if DATASET == "mvtec" else "Visa")

PRETRAINED_MODEL = "ViT-L/14@336px"
IMAGE_SIZE = 518
FEATURES_LIST = [6, 12, 18, 24]

K_SHOTS = 1
SEED = 10
BATCH_SIZE = 1
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# 직접 checkpoint를 지정하려면 문자열 경로를 넣으세요. None이면 로컬 후보에서 자동 탐색합니다.
CHECKPOINT_PATH = None
AUTO_DOWNLOAD_CHECKPOINT = False
HF_REPO_ID = "csgaobb/AdaptCLIP"

N_CTX = 12
VL_REDUCTION = 4
PQ_MID_DIM = 128
FUSION_TYPE = "average_mean"
SIGMA = 4

EVAL_METRICS = ["I-AUROC", "I-AP", "I-F1max", "P-AUROC", "P-AP", "P-F1max"]

# severity는 요청대로 3 고정
SEVERITY = 3
CORRUPTIONS = [
    "gaussian_noise",
    "motion_blur",
    "brightness",
    "contrast",
    "jpeg_compression",
    "downsample_upsample",
]
CORRUPTION = "gaussian_noise"  # 빠른 단일 실행용. 위 목록 중 하나로 바꾸세요.

CORRUPT_PROMPTS = False
SAVE_SELECTED_HEATMAPS = True

RESULT_ROOT = REPO / "result_adaptclip" / "corruption"
print("device:", DEVICE)
print("dataset path:", TEST_DATA_PATH)
print("target class:", TARGET_CLASS)
print("severity:", SEVERITY)


device: mps
dataset path: /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP/dataset/MVTec
target class: bottle
severity: 3


In [12]:
def _checkpoint_sort_key(path: Path) -> tuple[int, str]:
    text = str(path)
    expected = f"{N_CTX}_{VL_REDUCTION}_{PQ_MID_DIM}_train_on_mvtec_3adapters_batch8"
    score = 0
    if expected in text:
        score -= 20
    if path.name == "epoch_15.pth":
        score -= 10
    if DATASET in text:
        score -= 1
    return score, text


def _find_local_checkpoints() -> list[Path]:
    roots = [
        REPO / "adaptclip_checkpoints",
        REPO / "adaptclip_checkpoint",
        REPO / "checkpoints",
        REPO,
    ]
    found = []
    for root in roots:
        if root.exists():
            found.extend(root.glob("**/*.pth"))
            found.extend(root.glob("**/*.pt"))
    return sorted(set(found), key=_checkpoint_sort_key)


def download_checkpoints_from_hf() -> Path:
    try:
        from huggingface_hub import snapshot_download
    except ImportError as exc:
        raise ImportError("huggingface_hub가 필요합니다. `pip install huggingface-hub` 후 다시 실행하세요.") from exc

    local_dir = REPO / "adaptclip_checkpoints"
    local_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading checkpoints from {HF_REPO_ID} -> {local_dir}")
    snapshot_download(
        repo_id=HF_REPO_ID,
        local_dir=local_dir,
        allow_patterns=["*.pth", "*.pt", "*.bin", "*.safetensors"],
        local_dir_use_symlinks=False,
    )
    found = _find_local_checkpoints()
    if not found:
        raise FileNotFoundError(f"{HF_REPO_ID}에서 checkpoint 파일을 찾지 못했습니다: {local_dir}")
    return found[0]


def find_checkpoint() -> Path:
    if CHECKPOINT_PATH:
        path = Path(CHECKPOINT_PATH).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"CHECKPOINT_PATH가 없습니다: {path}")
        return path

    train_dataset = "mvtec"
    base_dir = f"{N_CTX}_{VL_REDUCTION}_{PQ_MID_DIM}_train_on_{train_dataset}_3adapters_batch8"
    candidates = [
        REPO / "adaptclip_checkpoints" / base_dir / "epoch_15.pth",
        REPO / "adaptclip_checkpoint" / base_dir / "epoch_15.pth",
        REPO / "adaptclip_checkpoints" / "epoch_15.pth",
        REPO / "adaptclip_checkpoint" / "epoch_15.pth",
    ]
    for path in candidates:
        if path.exists():
            return path

    found = _find_local_checkpoints()
    if found:
        print("local checkpoints:")
        for item in found[:10]:
            print(" -", item)
        return found[0]

    if AUTO_DOWNLOAD_CHECKPOINT:
        return download_checkpoints_from_hf()

    msg = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "pretrained checkpoint를 찾지 못했습니다. `AUTO_DOWNLOAD_CHECKPOINT=True`로 두거나 "
        "HuggingFace에서 받은 .pth 경로를 CHECKPOINT_PATH에 넣어주세요.\n\n찾아본 후보:\n" + msg
    )


CKPT = find_checkpoint()
print("checkpoint:", CKPT)

local checkpoints:
 - /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP/adaptclip_checkpoints/adaptclip_checkpoints/12_4_128_train_on_mvtec_3adapters_batch8/epoch_15.pth
 - /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP/adaptclip_checkpoints/adaptclip_checkpoints/12_4_128_train_on_visa_3adapters_batch8/epoch_15.pth
checkpoint: /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP/adaptclip_checkpoints/adaptclip_checkpoints/12_4_128_train_on_mvtec_3adapters_batch8/epoch_15.pth


In [13]:
def corruption_label(corruption: str, severity: int = SEVERITY) -> str:
    return f"{corruption}_s{severity}"


def build_command(corruption: str, severity: int = SEVERITY) -> tuple[list[str], Path]:
    label = corruption_label(corruption, severity)
    save_path = RESULT_ROOT / DATASET / TARGET_CLASS / label

    cmd = [
        sys.executable,
        "test.py",
        "--dataset", DATASET,
        "--class_name", TARGET_CLASS,
        "--test_data_path", str(TEST_DATA_PATH),
        "--seed", str(SEED),
        "--k_shots", str(K_SHOTS),
        "--checkpoint_path", str(CKPT),
        "--save_path", str(save_path),
        "--pretrained_model", PRETRAINED_MODEL,
        "--features_list", *map(str, FEATURES_LIST),
        "--image_size", str(IMAGE_SIZE),
        "--batch_size", str(BATCH_SIZE),
        "--n_ctx", str(N_CTX),
        "--vl_reduction", str(VL_REDUCTION),
        "--pq_mid_dim", str(PQ_MID_DIM),
        "--sigma", str(SIGMA),
        "--fusion_type", FUSION_TYPE,
        "--eval_metrics", *EVAL_METRICS,
        "--device", DEVICE,
        "--corruption", corruption,
        "--corruption_severity", str(severity),
        "--visual_learner",
        "--textual_learner",
        "--pq_learner",
        "--pq_context",
    ]

    if not SAVE_SELECTED_HEATMAPS:
        cmd += ["--no-save_selected_heatmaps"]
    if CORRUPT_PROMPTS:
        cmd += ["--corrupt_prompts"]

    return cmd, save_path


def parse_last_metric_table(log_path: Path) -> pd.DataFrame:
    text = log_path.read_text(errors="replace")
    lines = [line.strip() for line in text.splitlines()]
    table_lines = [line for line in lines if line.startswith("|") and line.endswith("|")]
    if not table_lines:
        return pd.DataFrame()

    start = 0
    for i, line in enumerate(table_lines):
        if "Name" in line:
            start = i
    table_lines = table_lines[start:]
    rows = []
    for line in table_lines:
        cols = [c.strip() for c in line.strip("|").split("|")]
        if all(re.fullmatch(r":?-{2,}:?", c.replace(" ", "")) for c in cols):
            continue
        rows.append(cols)
    if len(rows) < 2:
        return pd.DataFrame()
    df = pd.DataFrame(rows[1:], columns=rows[0])
    for col in df.columns:
        if col != "Name":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def run_one(corruption: str = CORRUPTION, severity: int = SEVERITY) -> dict:
    label = corruption_label(corruption, severity)
    cmd, save_path = build_command(corruption, severity)
    save_path.mkdir(parents=True, exist_ok=True)

    stdout_path = save_path / "stdout.txt"
    print(f"\n=== {TARGET_CLASS} / {label} ===")
    print(" ".join(shlex.quote(x) for x in cmd))
    started = time.time()
    proc = subprocess.run(cmd, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = time.time() - started
    stdout_path.write_text(proc.stdout, errors="replace")

    log_path = save_path / f"{DATASET}_{SEED}seed_{K_SHOTS}shot_test_log.txt"
    df = parse_last_metric_table(log_path) if log_path.exists() else pd.DataFrame()

    if proc.returncode != 0:
        print(proc.stdout[-8000:])
        raise RuntimeError(f"{TARGET_CLASS}/{label} 실패: returncode={proc.returncode}. full stdout: {stdout_path}")

    row = {"class": TARGET_CLASS, "corruption": label, "seconds": round(elapsed, 1), "log": str(log_path)}
    if not df.empty:
        metric_row = df[df["Name"].eq(TARGET_CLASS)].tail(1) if "Name" in df else pd.DataFrame()
        if not metric_row.empty:
            for col in metric_row.columns:
                if col != "Name":
                    row[col] = metric_row.iloc[-1][col]
    print(f"done in {elapsed:.1f}s -> {log_path}")
    if not df.empty:
        display(df)
    return row


## 단일 corruption 실행

`CORRUPTION`과 `TARGET_CLASS`만 바꿔서 한 번씩 돌리는 셀입니다.


In [14]:
one_result = run_one(CORRUPTION)
pd.DataFrame([one_result])



=== bottle / gaussian_noise_s3 ===
/usr/local/bin/python3 test.py --dataset mvtec --class_name bottle --test_data_path /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP/dataset/MVTec --seed 10 --k_shots 1 --checkpoint_path /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP/adaptclip_checkpoints/adaptclip_checkpoints/12_4_128_train_on_mvtec_3adapters_batch8/epoch_15.pth --save_path /Users/taehayeong/Desktop/Git_workspace/AdaptCLIP/result_adaptclip/corruption/mvtec/bottle/gaussian_noise_s3 --pretrained_model ViT-L/14@336px --features_list 6 12 18 24 --image_size 518 --batch_size 1 --n_ctx 12 --vl_reduction 4 --pq_mid_dim 128 --sigma 4 --fusion_type average_mean --eval_metrics I-AUROC I-AP I-F1max P-AUROC P-AP P-F1max --device mps --corruption gaussian_noise --corruption_severity 3 --visual_learner --textual_learner --pq_learner --pq_context


KeyboardInterrupt: 

## 같은 클래스에서 corruption 전체 실행


In [ ]:
rows = []
for corruption in CORRUPTIONS:
    rows.append(run_one(corruption, SEVERITY))

summary = pd.DataFrame(rows)
summary_path = RESULT_ROOT / DATASET / TARGET_CLASS / f"summary_seed{SEED}_{K_SHOTS}shot_s{SEVERITY}.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_path, index=False)
print("saved:", summary_path)
summary


## severity 3 고정

severity sweep은 제거했습니다. 이 노트북은 항상 `SEVERITY = 3`만 사용합니다.


In [ ]:
# severity는 3으로 고정되어 있습니다. 필요하면 위 설정 셀의 SEVERITY만 직접 바꾸세요.
